In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyArrowPatch

%matplotlib inline

In [ ]:
# ── Data ──────────────────────────────────────────────────────────────────────
epsilon = np.array([0, 0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0])

plots = [
    dict(
        title="Instable trade routes (chokepoints)",
        upper=np.array([20, 22.5, 24.5, 26.0, 27.2, 28.1, 28.8, 29.4, 29.8]),
        lower=np.array([20, 17.5, 15.5, 14.0, 12.8, 11.9, 11.2, 10.6, 10.2]),
        y_max=200,
        y_mid=20,           # curves originate at 20
        color="#185FA5",
        alpha=0.25,
        arrow_up=None,
        arrow_dn=["Decreased", "high-risk routes"],
        y_label_add="through trade chokepoints",
        show_upper=False,
        show_lower=True,
        # Regional decomposition for lower curve
        regions=[
            dict(name="Malacca", values=np.array([8, 7, 6.5, 6.0, 5.5, 5.2, 4.8, 4.5, 4.2]), color="#1f77b4", linestyle="-", alpha=0.6),
            dict(name="Hormuz", values=np.array([8, 8, 8, 8, 8, 8, 8, 8, 8]), color="#ff7f0e", linestyle="-", alpha=0.6),
            dict(name="Panama", values=np.array([4, 2.5, 1.5, 0.0, -0.7, -1.2, -1.6, -1.9, -2.2]), color="#2ca02c", linestyle="--", alpha=0.6),
        ]
    ),
    dict(
        title="Stable exporters",
        upper=np.array([100, 118, 133, 147, 158, 167, 175, 182, 188]),
        lower=np.array([100,  82,  67,  54,  43,  34,  27,  21,  16]),
        y_max=200,
        y_mid=100,
        color="#3B6D11",
        alpha=0.25,
        arrow_up=["Increased stable", "HBI producers"],
        arrow_dn=None,
        y_label_add="in stable regions",
        show_upper=True,
        show_lower=False,
        # Regional decomposition for upper curve
        regions=[
            dict(name="South America", values=np.array([50, 62, 75, 87, 97, 106, 113, 119, 124]), color="#d62728", linestyle="-", alpha=0.6),
            dict(name="Oceania", values=np.array([50, 56, 58, 60, 61, 61, 62, 63, 64]), color="#9467bd", linestyle="-", alpha=0.6),
            dict(name="North America", values=np.array([20, 15, 12, 10, 8, 6, 5, 4, 3]), color="#8c564b", linestyle="--", alpha=0.6),
            dict(name="Europe", values=np.array([15, 12, 10, 8, 6, 4, 3, 2, 1]), color="#e377c2", linestyle="--", alpha=0.6),
            dict(name="Asia", values=np.array([25, 20, 15, 10, 8, 5, 3, 2, 1]), color="#bcbd22", linestyle="--", alpha=0.6),
        ]
    ),
    dict(
        title="Fragmentation/Block trade",
        upper=np.array([37.5, 44, 50, 55, 59, 63, 66, 69, 71]),
        lower=np.array([37.5, 31, 25, 20, 16, 13, 10,  8,  6]),
        y_max=200,
        y_mid=37.5,
        color="#534AB7",
        alpha=0.25,
        arrow_up=None,
        arrow_dn=["Decreased trade", "between blocks"],
        y_label_add="between Block A and B",
        show_upper=False,
        show_lower=True,
        # Regional decomposition for lower curve
        regions=[
            dict(name="China", values=np.array([20, 17, 14, 11, 9, 7, 5, 4, 3]), color="#17becf", linestyle="-", alpha=0.6),
            dict(name="Oceania", values=np.array([17.5, 14, 11, 9, 7, 6, 5, 4, 3]), color="#aec7e8", linestyle="-", alpha=0.6),
        ]
    ),
]

# ── Layout ────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.subplots_adjust(wspace=0.35)

ARROW_X   = 4.5          # x position of arrows (data coords)
LABEL_X   = 4.75         # x position of annotation text (data coords)
GAP_FRAC  = 0.04         # gap between arrow start and y_mid, as fraction of y_max
SMOOTH_N  = 300          # points for smooth interpolation

# ── Smooth interpolation helper ───────────────────────────────────────────────
def smooth(x, y, n=SMOOTH_N):
    from scipy.interpolate import make_interp_spline
    x_new = np.linspace(x[0], x[-1], n)
    spl   = make_interp_spline(x, y, k=3)
    return x_new, spl(x_new)

# ── Draw each subplot ─────────────────────────────────────────────────────────
for ax, p in zip(axes, plots):
    gap = p["y_max"] * GAP_FRAC

    # Smooth curves
    ex_up, ey_up = smooth(epsilon, p["upper"])
    ex_dn, ey_dn = smooth(epsilon, p["lower"])

    # Filled area between curves (only if both are shown)
    if p["show_upper"] and p["show_lower"]:
        ax.fill_between(ex_up, ey_up, ey_dn, color=p["color"], alpha=p["alpha"])

    # Lines + markers (conditional on show_upper/show_lower)
    if p["show_upper"]:
        ax.plot(ex_up, ey_up, color=p["color"], linewidth=2.5, label="Global", zorder=10)
        ax.plot(epsilon, p["upper"], "o", color=p["color"], markersize=4, zorder=11)
    
    if p["show_lower"]:
        ax.plot(ex_dn, ey_dn, color=p["color"], linewidth=2.5, label="Global", zorder=10)
        ax.plot(epsilon, p["lower"], "o", color=p["color"], markersize=4, zorder=11)

    # Plot regional decomposition lines
    if "regions" in p and p["regions"]:
        for region in p["regions"]:
            ex_reg, ey_reg = smooth(epsilon, region["values"])
            ax.plot(ex_reg, ey_reg, color=region["color"], linewidth=1.2, 
                   linestyle=region["linestyle"], alpha=region["alpha"], 
                   label=region["name"], zorder=5)

    # Axes formatting
    ax.set_xlim(0, 8)
    ax.set_ylim(0, p["y_max"])
    ax.set_xlabel("ε in %", fontsize=11)
    ax.set_ylabel(f"HBI trade/production {p["y_label_add"]} in Mt", fontsize=10)
    ax.set_title(p["title"], fontsize=13, fontweight="medium", pad=10)
    ax.grid(True, linestyle="--", linewidth=0.5, alpha=0.5)
    ax.set_xticks([0, 2, 4, 6, 8])
    
    # Add legend for regional lines
    if "regions" in p and p["regions"]:
        ax.legend(loc="best", fontsize=7, framealpha=0.9)

    # ── Arrows (data coords, drawn in axis transform) ──────────────────────────
    y_up_end = p["upper"][-1]
    y_dn_end = p["lower"][-1]
    y_up_start = p["y_mid"] + gap
    y_dn_start = p["y_mid"] - gap

    arrowprops = dict(
        arrowstyle="->",
        color="#444441",
        lw=1.4,
        mutation_scale=12,
    )

    # Upper arrow (pointing up toward curve) - only if arrow_up is defined
    if p["arrow_up"] is not None:
        ax.annotate(
            "", xy=(ARROW_X, y_up_end), xytext=(ARROW_X, y_up_start),
            xycoords="data", textcoords="data",
            arrowprops=arrowprops,
            clip_on=False,
        )

    # Lower arrow (pointing down toward curve) - only if arrow_dn is defined
    if p["arrow_dn"] is not None:
        ax.annotate(
            "", xy=(ARROW_X, y_dn_end), xytext=(ARROW_X, y_dn_start),
            xycoords="data", textcoords="data",
            arrowprops=arrowprops,
            clip_on=False,
        )

    # ── Arrow labels ──────────────────────────────────────────────────────────
    y_label_up = (y_up_start + y_up_end) / 2
    y_label_dn = (y_dn_start + y_dn_end) / 2

    if p["arrow_up"] is not None:
        ax.text(
            LABEL_X, y_label_up,
            "\n".join(p["arrow_up"]),
            fontsize=8, va="center", ha="left",
            color="#444441",
            clip_on=False,
        )
    
    if p["arrow_dn"] is not None:
        ax.text(
            LABEL_X, y_label_dn,
            "\n".join(p["arrow_dn"]),
            fontsize=8, va="center", ha="left",
            color="#444441",
            clip_on=False,
        )

# plt.savefig("mga_plots.png", dpi=150, bbox_inches="tight")
plt.show()
# print("Saved to mga_plots.png")